#### **Generative Adverserial Network**

A Generative Adversarial Network (GAN) is a machine learning framework where two neural networks—a Generator and a Discriminator—compete in a zero-sum game. The generator creates synthetic data (like images), and the discriminator tries to tell if it's real or fake, causing both to continuously improve.

#### **A schematic GAN implementation**

- A generator maps vectors of shape latent dimension to images.
- A discriminator network maps images to a binary score estimating the probability that the image is real.
- A gan network chains the generator and the discriminator together. Thus this gan network maps latent space vectors to the dicriminator's asessment of the realism of these latent vectors as decoded by the generator.
- We train the discriminator using examples of real and fake images along with labels, just as we train any regular image-classification model.
- To train the generatort, we use the gradients of the generator's weights with regard to the loss of the gan model. This means, at everys step, we move the weights of the generator in a direction that makes the discriminator more likely to classify  as "real" the images decoded by the generator. In other words we train the generator to fool the discriminator.

#### **A bag of tricks**

- We use `tanh` as the last activation in the generator, instead of `sigmoid`, which is mpre commonly found in other types of models.
- We sample points from the latent space using a Gaussian distribution, not a uniform distribution.
- Stochasticity is good to induce robustness. Because GAN training results in a dynamic equilibrium. GANs are likely to get stuck in all sorts of ways. Introducing randomness during training helps prevent this. We introduce randomness in two ways: by using dropout in the discriminator and by random noise to the labels for the discriminator.
- Sparse gradients can hinder GAN training. In deep learning, sparsity s often a desirable property, but not in GANs. Two things can induce gradient sparsity: max pooling and `ReLu` activations, Instead pf max pooling, we use strided convolutions for downsampling, and we use a `LeakyReLU` layer instead of a `ReLU` activation. It's similar to `ReLU`, but it relaxes sparsity constraints by allowing samll negative activation values.
- In generated images, it's common to see checkerboard artifacts caused by unequal coverage of the pixel space in the generator. To fix this, we use a kernel size that's divisible by the stride size whenever we use a strided convolution

#### **The generator**

Only one problem here. The generator gets stuck with generated images that look like noise. A possible solution is to use dropout.

In [1]:
import keras
from keras import layers
import numpy as np

latent_dim = 1
height = 32
width = 32
channels = 1

generator_input = keras.Input(shape=(latent_dim,))

# Tranforms the input into a 16 x 16, 128-channel feature map
x = layers.Dense(128 * 16 * 16)(generator_input)
x = layers.LeakyReLU()(x)
x = layers.Reshape((16, 16, 128))(x)

x = layers.Conv2D(256, 5, padding="same")(x)
x = layers.LeakyReLU()(x)

# upsamples to 32 x 32
x = layers.Conv2DTranspose(256, 4, strides=2, padding="same")(x)
x = layers.LeakyReLU()(x)

x = layers.Conv2D(256, 5, padding="same")(x)
x = layers.LeakyReLU()(x)
x = layers.Conv2D(256, 5, padding="same")(x)
x = layers.LeakyReLU()(x)

x = layers.Conv2D(channels, 7, activation="tanh", padding="same")(x) # produce a 32 x 32 1-channel feature map
generator = keras.models.Model(generator_input, x) # instantiate the generator model
generator.summary()

2026-07-06 13:08:20.961356: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-07-06 13:08:20.961378: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 1)]               0         
                                                                 
 dense (Dense)               (None, 32768)             65536     
                                                                 
 leaky_re_lu (LeakyReLU)     (None, 32768)             0         
                                                                 
 reshape (Reshape)           (None, 16, 16, 128)       0         
                                                                 
 conv2d (Conv2D)             (None, 16, 16, 256)       819456    
                                                                 
 leaky_re_lu_1 (LeakyReLU)   (None, 16, 16, 256)       0         
                                                                 
 conv2d_transpose (Conv2DTra  (None, 32, 32, 256)      104883

2026-07-06 13:08:22.753639: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-07-06 13:08:22.753720: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublas.so.11'; dlerror: libcublas.so.11: cannot open shared object file: No such file or directory
2026-07-06 13:08:22.753781: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublasLt.so.11'; dlerror: libcublasLt.so.11: cannot open shared object file: No such file or directory
2026-07-06 13:08:22.753831: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcufft.so.10'; dlerror: libcufft.so.10: cannot open shared object file: No such file or directory
2026-07-06 13:08:22.803025: W tensorflow/stream_executor/platform/default/dso_loader.cc:64

#### **The Discriminator**

In [2]:
discriminator_input = layers.Input(shape=(height, width, channels))

x = layers.Conv2D(128, 3, padding="same")(discriminator_input)
x = layers.LeakyReLU()(x)

x = layers.Conv2D(128, 4, strides=2, padding="same")(x)
x = layers.LeakyReLU()(x)

x = layers.Conv2D(128, 4, strides=2, padding="same")(x)
x = layers.LeakyReLU()(x)

x = layers.Conv2D(128, 4, strides=2, padding="same")(x)
x = layers.LeakyReLU()(x)

x = layers.Flatten()(x)

x = layers.Dropout(0.4)(x)

x = layers.Dense(1, activation="sigmoid")(x) # produce a single score
discriminator = keras.models.Model(discriminator_input, x) # instantiate the discriminator model
discriminator.summary()

discriminator_optimizer = keras.optimizers.RMSprop(learning_rate=0.0008, clipvalue=1.0, decay=1e-8)
discriminator.compile(optimizer=discriminator_optimizer, loss="binary_crossentropy")

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 32, 32, 1)]       0         
                                                                 
 conv2d_4 (Conv2D)           (None, 32, 32, 128)       1280      
                                                                 
 leaky_re_lu_5 (LeakyReLU)   (None, 32, 32, 128)       0         
                                                                 
 conv2d_5 (Conv2D)           (None, 16, 16, 128)       262272    
                                                                 
 leaky_re_lu_6 (LeakyReLU)   (None, 16, 16, 128)       0         
                                                                 
 conv2d_6 (Conv2D)           (None, 8, 8, 128)         262272    
                                                                 
 leaky_re_lu_7 (LeakyReLU)   (None, 8, 8, 128)         0   

#### **The Adversarial Network**

To set up the GAN, connect the generator and discriminator. The model trains by adjusting the generator to better deceive the discriminator, classifying images as “fake” or “real” based on consistent labeling of input images as “real.” During training, the discriminator’s weights remain frozen and non-trainable to prevent it from learning to always predict “real,” which would undermine the training process.

In [3]:
discriminator.trainable = False

gan_input = keras.Input(shape=(latent_dim,))
gan_output = discriminator(generator(gan_input))
gan = keras.models.Model(gan_input, gan_output)

gan_optimizer = keras.optimizers.RMSprop(learning_rate=0.0004, clipvalue=1.0, decay=1e-8)
gan.compile(optimizer=gan_optimizer, loss="binary_crossentropy")

### **How to train DCGAN**

- Draw random points in the latent space (random noise).
- Generate images with generator using this random noise.
- Mix the generated images with real ones.
- Train `discriminator` using these mixed images, with corresponding targets: either “real” (for the real images) or “fake” (for the generated images).
- Draw new random points in the latent space.
- Train gan using these random vectors, with targets that all say “these are real images.” This updates the weights of the generator (only, because the discriminator is frozen inside gan) to move them toward getting the discriminator to predict “these are real images” for generated images: this trains the generator to fool the discriminator.

In [10]:
### Implementing GAN training

import os
from keras.preprocessing import image
from keras.datasets import fashion_mnist
import tensorflow as tf

(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

x_train = x_train[y_train.flatten() == 5]
x_train = np.pad(x_train[:,], (2, 2), 'constant')

x_train = x_train.reshape((x_train.shape[0],) + (height, width, channels)).astype("float32") / 255 # rescale the images to the [0, 1] range

iterations = 10000
batch_size = 20
save_dir = "./"

start = 0
for step in range(iterations):
    # Sample random points in the latent space
    random_latent_vectors = np.random.normal(size=(batch_size, latent_dim))

    # Decode them to fake images
    generated_images = generator.predict(random_latent_vectors)

    # Combine them with real images
    stop = start + batch_size
    real_images = x_train[start: stop]
    combined_images = np.concatenate([generated_images, real_images])

    # Assemble labels discriminating real from fake images
    labels = np.concatenate([np.ones((batch_size, 1)), np.zeros((batch_size, 1))])
    # Add random noise to the labels - important trick!
    labels += 0.05 * np.random.random(labels.shape)

    # Train the discriminator
    d_loss = discriminator.train_on_batch(combined_images, labels)

    # Sample random points in the latent space
    random_latent_vectors = np.random.normal(size=(batch_size, latent_dim))
    # Assemble labels that say "all real images"
    misleading_targets = np.zeros((batch_size, 1))
    # Train the generator (note that we should *not* update the weights of the discriminator)!
    a_loss = gan.train_on_batch(random_latent_vectors, misleading_targets)

    start += batch_size
    if start > len(x_train) - batch_size:
        start = 0

    # Occasionally save the weights and generate images
    if step % 100 == 0:
        gan.save_weights(os.path.join(save_dir, "gan.h5"))
        print("discriminator loss:", d_loss)
        print("adversarial loss:", a_loss)
        # Save one generated image:
        img = tf.keras.utils.array_to_img(generated_images[0] * 255., scale=False)
        img.save(os.path.join(save_dir, 'generated_horse' + str(step) + ".png"))

        img = tf.keras.utils.array_to_img(real_images[0] * 255., scale=False)
        img.save(os.path.join(save_dir, 'real_horse' + str(step) + ".png"))

1/1 [==============================] - 0s 322ms/step
discriminator loss: 2.149742841720581
adversarial loss: 0.7566272616386414
1/1 [==============================] - 0s 320ms/step
discriminator loss: 0.41081205010414124
adversarial loss: 1.3992931842803955
1/1 [==============================] - 0s 311ms/step
discriminator loss: 0.2502109408378601
adversarial loss: 1.6901315450668335
1/1 [==============================] - 0s 307ms/step
discriminator loss: 0.5968325734138489
adversarial loss: 2.1599974632263184
1/1 [==============================] - 0s 315ms/step
discriminator loss: 0.6410516500473022
adversarial loss: 1.4512962102890015
1/1 [==============================] - 0s 307ms/step
discriminator loss: 0.36950671672821045
adversarial loss: 10.238247871398926
1/1 [==============================] - 0s 312ms/step
discriminator loss: 0.07174091786146164
adversarial loss: 5.8314900398254395
1/1 [==============================] - 0s 352ms/step
discriminator loss: 0.11896133422851562
ad